In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import matplotlib.dates as mdates
from sklearn import datasets, linear_model
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
import statsmodels.api as sm
from scipy.stats import t
from scipy.optimize import minimize
import os
import cvxpy as cp
from tqdm import tqdm
import seaborn as sns
import torch

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import openpyxl

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.autograd import Function

from sklearn.preprocessing import StandardScaler

from functions import CVaRSolver

import functions

# Data Preparation

Do following for different risk aversions beta:

for date in test_dates:

1. prepare train and test data (date)

2. prepare scenario matrix out of train data

3. specify constants for cvxpy (dimensions of scenario matrix, etc.)

4. precompute oracle solution w*(c)

5. Train VAR on train with custom loss function (combinations of MSE and DFL loss)

    Do so with every combination
    Save relevant metrics
    Estimate returns c_hat and w*(c_hat)

-> Agreggate metrics ofer the whole test period (backtesting period)

In [2]:
data_path = "../data/processed/"

return_data = pd.read_csv(data_path + "stationary_return_data_subset.csv")

In [3]:
# Pivot return data
return_matrix = return_data.pivot(index='date', columns='RIC', values='return')
display(return_matrix)

RIC,BRO.N,CTRA.N,CVS.N,FFIV.OQ,HSY.N,LLY.N,NEM.N,PCG.N,REGN.OQ,TYL.N
date,,,,,,,,,,
2000-01-31,-0.104405,-0.081712,-1.236885e-01,-0.175439,-0.105263,0.005639,-0.168367,7.012195e-02,-0.034314,-0.204545
2000-02-29,-0.035086,0.074857,1.788909e-03,-0.042553,0.039651,-0.107465,0.085890,-5.982906e-02,3.588832,0.228571
2000-03-31,0.172348,0.142292,7.321429e-02,-0.247222,0.109531,0.059937,0.015535,3.281437e-02,-0.476770,0.104651
2000-04-28,0.037157,0.027682,1.595897e-01,-0.310886,-0.069231,0.227183,0.044568,2.351190e-01,-0.033827,-0.094737
2000-05-31,0.165747,0.345877,1.398903e-11,-0.309237,0.148974,-0.011941,-0.016000,1.444134e-11,-0.286652,-0.255814
...,...,...,...,...,...,...,...,...,...,...
2023-09-29,-0.057490,-0.040440,7.135185e-02,-0.015398,-0.068789,-0.030801,-0.052920,-1.042945e-02,-0.004271,-0.030846
2023-10-31,-0.004152,0.016636,-3.132471e-03,-0.059265,-0.063625,0.031277,0.014073,1.053937e-02,-0.052335,-0.034288
2023-11-30,0.076635,-0.038423,-1.536009e-02,0.129296,0.009148,0.068968,0.083216,5.337423e-02,0.056316,0.096380


In [4]:
X, Y = functions.create_time_series_data_with_lags(return_matrix, max_lag=3)

print(X.shape)
print(Y.shape)

(286, 30)
(286, 10)


## XX

In [19]:
# Code it for the last day as test day

test_index = 1 # index of the test day, counting from the end of the dataset (1 means the last day, 2 means the second to last day, etc.)

alpha = 0.95 # CVaR confidence level (95% confidence level means we are looking at the worst 5% of cases)
beta = 0.08 # CVaR risk aversion (average loss in percentage of initial investment in the worst 5% of cases)

In [20]:
X_train = X[:-test_index] # take all rows up to the test_index last row for training
X_test = X[-test_index] # take the test_index last row for testing

Y_train = Y[:-test_index]
Y_test = Y[-test_index]

len(Y_train), len(Y_test)

(285, 10)

In [21]:
# Perform bootstrap for return scenarios on the training period

return_array = return_matrix[:-test_index].values

# Number of bootstrap scenarios
num_scenarios = 1000

# Set random seed for reproducibility
np.random.seed(42)

# Sample row indices with replacement
sample_indices = np.random.choice(return_array.shape[0], size=num_scenarios, replace=True)

# Generate bootstrapped scenario matrix
scenario_matrix = return_array[sample_indices, :]

loss_matrix = -scenario_matrix
# scenario_matrix (S scenarios × N assets)

S, N = scenario_matrix.shape

# Empirical mean from scenarios
mu = scenario_matrix.mean(axis=0)

print(S, N)

1000 10


In [22]:
# instantiate general CVaR problem solver

solver = CVaRSolver(
    loss_matrix=loss_matrix,
    N=N,
    S=S,
    alpha=alpha,
    beta=beta
)

In [12]:
# precompute oracle solutions for training
# for every sample in train set we need the exact solution

# Precompute oracle solutions
oracle_solutions = [
    solver.solve(mu).copy()
    for mu in Y_train
]

In [23]:
# Convert to tensor
oracle_tensor = torch.tensor(
    np.array(oracle_solutions),
    dtype=torch.float32
)

# Prepare dataset for pytorch training
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train = x_scaler.fit_transform(X_train)
Y_train = y_scaler.fit_transform(Y_train)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)

# Include in dataset — index alignment is preserved through shuffling
train_dataset = TensorDataset(
    X_train_tensor,
    Y_train_tensor,
    oracle_tensor
)

In [24]:
batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True        # TensorDataset shuffles all tensors together
)

In [25]:
class VARasNN(nn.Module):

    def __init__(self,input_dim,output_dim):

        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            output_dim
        )

    def forward(self,x):

        return self.linear(x)

In [28]:
input_dim = X_train.shape[1]
output_dim = Y_train.shape[1]

model = VARasNN(input_dim, output_dim)

print(model)

VARasNN(
  (linear): Linear(in_features=30, out_features=10, bias=True)
)


In [29]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [27]:
gamma = 0.5  # 0 = full MSE, 1 = full SPO+

criterion = nn.MSELoss()

n_epochs = 5

for epoch in tqdm(range(n_epochs), desc="epochs"):

    model.train()
    epoch_loss = 0

    for X_batch, Y_batch, oracle_batch in tqdm(train_loader, desc="batches"):

        optimizer.zero_grad()

        predictions = model(X_batch)

        # SPO+ loss
        spo_losses = []
        for i in range(X_batch.shape[0]):
            loss_i = SPOPlus.apply(
                predictions[i],
                Y_batch[i],
                oracle_batch[i],
                solver
            )
            spo_losses.append(loss_i)
        spo_loss = torch.stack(spo_losses).mean()

        # MSE loss
        mse_loss = criterion(predictions, Y_batch)

        # combined loss
        loss = gamma * spo_loss + (1 - gamma) * mse_loss

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1}: avg SPO+ loss = {avg_loss:.6f}")

epochs:   0%|          | 0/5 [00:00<?, ?it/s]

epochs:  20%|██        | 1/5 [00:41<02:44, 41.10s/it]

Epoch 1: avg SPO+ loss = 0.613006


epochs:  40%|████      | 2/5 [01:23<02:05, 41.94s/it]

Epoch 2: avg SPO+ loss = 0.609874


...

Epoch 3: avg SPO+ loss = 0.612660


epochs:  80%|████████  | 4/5 [02:42<00:39, 39.59s/it]

Epoch 4: avg SPO+ loss = 0.621731


epochs: 100%|██████████| 5/5 [03:17<00:00, 39.51s/it]

Epoch 5: avg SPO+ loss = 0.610191


In [30]:
for epoch in tqdm(range(n_epochs), desc="epochs"):

    model.train()
    epoch_spo_loss = 0
    epoch_mse_loss = 0
    epoch_combined_loss = 0

    for X_batch, Y_batch, oracle_batch in train_loader:

        optimizer.zero_grad()
        predictions = model(X_batch)

        spo_losses = []
        for i in range(X_batch.shape[0]):
            loss_i = SPOPlus.apply(
                predictions[i],
                Y_batch[i],
                oracle_batch[i],
                solver
            )
            spo_losses.append(loss_i)
        spo_loss = torch.stack(spo_losses).mean()
        mse_loss = criterion(predictions, Y_batch)
        loss = gamma * spo_loss + (1 - gamma) * mse_loss

        loss.backward()
        optimizer.step()

        epoch_spo_loss      += spo_loss.item()
        epoch_mse_loss      += mse_loss.item()
        epoch_combined_loss += loss.item()

    n = len(train_loader)
    print(
        f"Epoch {epoch+1}: "
        f"combined={epoch_combined_loss/n:.6f} | "
        f"SPO+={epoch_spo_loss/n:.6f} | "
        f"MSE={epoch_mse_loss/n:.6f}"
    )

epochs:  20%|██        | 1/5 [00:31<02:06, 31.61s/it]

Epoch 1: combined=0.567756 | SPO+=-0.238094 | MSE=1.373606


epochs:  40%|████      | 2/5 [01:02<01:34, 31.37s/it]

Epoch 2: combined=0.553891 | SPO+=-0.261149 | MSE=1.368931


epochs:  60%|██████    | 3/5 [01:35<01:03, 31.87s/it]

Epoch 3: combined=0.552292 | SPO+=-0.258078 | MSE=1.362661


epochs:  80%|████████  | 4/5 [02:05<00:31, 31.22s/it]

Epoch 4: combined=0.528543 | SPO+=-0.270132 | MSE=1.327217


epochs: 100%|██████████| 5/5 [02:38<00:00, 31.71s/it]

Epoch 5: combined=0.522194 | SPO+=-0.281839 | MSE=1.326228


In [ ]:
# model.eval()

# with torch.no_grad():

#     preds_scaled = model(X_test_tensor).numpy()

# preds = y_scaler.inverse_transform(preds_scaled)

In [ ]:
# preds = y_scaler.inverse_transform(preds_scaled)
# Y_true = y_scaler.inverse_transform(Y_test_tensor.numpy())

In [ ]:
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# mse = mean_squared_error(Y_true, preds)
# mae = mean_absolute_error(Y_true, preds)
# r2  = r2_score(Y_true, preds)

# print("GLOBAL STATS")
# print("MSE:", mse)
# print("MAE:", mae)
# print("R2 :", r2)

GLOBAL STATS
MSE: 0.006941961590200663
MAE: 0.06680236756801605
R2 : -0.7894545793533325
